# CertGraph: Certificate-Aware Heterogeneous GNN for ADCS Vulnerability Classification
## Multi-Class ESC Attack Path Detection in Active Directory

---

**Novel Contribution (Phase 2)**: First GNN-based multi-class ADCS vulnerability classifier that models 
certificate templates, CAs, and issuance policies as first-class heterogeneous graph nodes.

**Classes**: ESC1 | ESC2 | ESC3 | ESC4 | ESC9 | ESC13 | Safe

**Evaluation**: 5-fold cross-validation on 700 synthetic AD environments with 4 baselines.


## 1. Theoretical Background

### 1.1 ESC Vulnerability Taxonomy

| ESC | Vulnerability | Key Indicator | Graph Condition |
|-----|---------------|---------------|-----------------|
| ESC1 | Enrollee supplies subject + Client Auth | `CT_FLAG_ENROLLEE_SUPPLIES_SUBJECT` | Low-priv users have enrollment rights |
| ESC2 | Any Purpose EKU | `OID 2.5.29.37.0` | Low-priv users have enrollment rights |
| ESC3 | Certificate Request Agent + RA signature | `OID 1.3.6.1.4.1.311.20.2.1` | Low-priv users have enrollment rights |
| ESC4 | Vulnerable template ACLs | Writable ACEs on template | Low-priv users have WriteDacl/GenericAll |
| ESC9 | No security extension flag + Client Auth | `CT_FLAG_NO_SECURITY_EXTENSION` | Low-priv users have enrollment rights |
| ESC13 | Issuance policy OID linked to group | `msPKI-Certificate-Policy` → Group | Group contains vulnerable users |
| Safe | No exploitable misconfiguration | None of the above | Safe configuration OR secure permissions |

### 1.2 Adversarial Hard Negatives

Flat classifiers (MLP, RF) look only at template configuration flags. In practice, a template with unsafe flags is only exploitable if target edge relations (e.g. low-priv enrollment, write permissions) exist in the graph. We generate **Hard Negatives** (Safe templates with vulnerable flags but secure permissions) to test if models can utilize graph structure.


## 2. Environment Setup


In [ ]:
import json, os, time, random, warnings, sys
sys.path.append(os.path.abspath(os.path.join("..")))
sys.path.append(os.path.abspath(os.path.join("..", "src")))
sys.path.append(os.path.abspath(os.path.join("..", "src", "phase2_certgraph")))
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import GATConv, HeteroConv
from torch_geometric.loader import DataLoader
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report,
    confusion_matrix as sk_confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.manifold import TSNE
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
%matplotlib inline

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({
    "figure.facecolor": "#1a1a2e", "axes.facecolor": "#16213e",
    "text.color": "white", "axes.labelcolor": "#aaa",
    "xtick.color": "#666", "ytick.color": "#666",
})
print(f"PyTorch: {torch.__version__}")


## 3. Synthetic Dataset Generator


In [ ]:
import json
import os
import random
import numpy as np
import torch
from torch_geometric.data import HeteroData

# ── EKU OIDs ──
EKU_CLIENT_AUTH = "1.3.6.1.5.5.7.3.2"
EKU_ANY_PURPOSE = "2.5.29.37.0"
EKU_CERT_REQUEST_AGENT = "1.3.6.1.4.1.311.20.2.1"
EKU_CODE_SIGNING = "1.3.6.1.5.5.7.3.3"
EKU_EMAIL_PROTECTION = "1.3.6.1.5.5.7.3.4"
EKU_EFS = "1.3.6.1.4.1.311.10.3.4"
EKU_SERVER_AUTH = "1.3.6.1.5.5.7.3.1"

# ── Flag constants ──
CT_FLAG_ENROLLEE_SUPPLIES_SUBJECT = 0x1
CT_FLAG_NO_SECURITY_EXTENSION = 0x80000

ESC_CLASSES = ["ESC1", "ESC2", "ESC3", "ESC4", "ESC9", "ESC13", "Safe"]
NUM_CLASSES = len(ESC_CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(ESC_CLASSES)}


# ─────────────────────────────────────────────────────────────────────
# Template generators — one per ESC class
# ─────────────────────────────────────────────────────────────────────

def _gen_esc1_template() -> dict:
    """ESC1: Enrollee supplies subject + Client Auth + no approval + no RA sig."""
    return {
        "name_flag": CT_FLAG_ENROLLEE_SUPPLIES_SUBJECT,
        "enrollment_flag": 0,
        "ra_signature": 0,
        "ekus": [EKU_CLIENT_AUTH],
        "has_issuance_policy_oid": False,
        "has_vulnerable_acl": False,
        "schema_version": 2,
        "hard_negative_type": None,
    }


def _gen_esc2_template() -> dict:
    """ESC2: Any Purpose EKU or SubCA + no manager approval."""
    return {
        "name_flag": random.choice([0, 0x2000000]),
        "enrollment_flag": random.choice([0, 32]),
        "ra_signature": 0,
        "ekus": [EKU_ANY_PURPOSE],
        "has_issuance_policy_oid": False,
        "has_vulnerable_acl": False,
        "schema_version": 2,
        "hard_negative_type": None,
    }


def _gen_esc3_template() -> dict:
    """ESC3: Certificate Request Agent EKU + RA sig on dependent template."""
    return {
        "name_flag": random.choice([0, 0x2000000]),
        "enrollment_flag": random.choice([0, 32]),
        "ra_signature": 1,
        "ekus": [EKU_CERT_REQUEST_AGENT] if random.random() < 0.5 else [EKU_CLIENT_AUTH],
        "has_issuance_policy_oid": False,
        "has_vulnerable_acl": False,
        "schema_version": 2,
        "hard_negative_type": None,
    }


def _gen_esc4_template() -> dict:
    """ESC4: Template has writable ACEs by low-priv principals."""
    return {
        "name_flag": random.choice([0, 0x2000000]),
        "enrollment_flag": random.choice([0, 32, 43]),
        "ra_signature": random.choice([0, 1]),
        "ekus": [random.choice([EKU_CODE_SIGNING, EKU_CLIENT_AUTH, EKU_SERVER_AUTH])],
        "has_issuance_policy_oid": False,
        "has_vulnerable_acl": True,
        "schema_version": 2,
        "hard_negative_type": None,
    }


def _gen_esc9_template() -> dict:
    """ESC9: No security extension flag + Client Auth."""
    return {
        "name_flag": 0x2000000,
        "enrollment_flag": CT_FLAG_NO_SECURITY_EXTENSION | random.choice([0, 9]),
        "ra_signature": 0,
        "ekus": [EKU_CLIENT_AUTH] + random.sample([EKU_EFS, EKU_EMAIL_PROTECTION], k=random.randint(0, 2)),
        "has_issuance_policy_oid": False,
        "has_vulnerable_acl": False,
        "schema_version": 2,
        "hard_negative_type": None,
    }


def _gen_esc13_template() -> dict:
    """ESC13: Issuance policy OID linked to group + Client Auth."""
    return {
        "name_flag": 0x2000000,
        "enrollment_flag": 0,
        "ra_signature": 0,
        "ekus": [EKU_CLIENT_AUTH],
        "has_issuance_policy_oid": True,
        "has_vulnerable_acl": False,
        "schema_version": 2,
        "hard_negative_type": None,
    }


def _gen_safe_template() -> dict:
    """
    Safe: Non-vulnerable template.
    Adversarial Modification: 50% of Safe templates are generated as 'Hard Negatives'
    that look like vulnerabilities based on configuration flags, but lack the critical
    graph permissions (edges) required for exploitation.
    """
    mode = random.choice(["Normal", "HN_ESC1", "HN_ESC4", "HN_ESC13"])
    if mode == "Normal":
        return {
            "name_flag": 0x2000000,  # no enrollee-supplies-subject
            "enrollment_flag": random.choice([0, 2, 32]),  # may require approval
            "ra_signature": random.choice([0, 1]),
            "ekus": [random.choice([EKU_CODE_SIGNING, EKU_SERVER_AUTH, EKU_EMAIL_PROTECTION])],
            "has_issuance_policy_oid": False,
            "has_vulnerable_acl": False,
            "schema_version": random.choice([1, 2, 4]),
            "hard_negative_type": None,
        }
    elif mode == "HN_ESC1":
        # Features match ESC1, but we will block enrollment edges for low-priv users
        return {
            "name_flag": CT_FLAG_ENROLLEE_SUPPLIES_SUBJECT,
            "enrollment_flag": 0,
            "ra_signature": 0,
            "ekus": [EKU_CLIENT_AUTH],
            "has_issuance_policy_oid": False,
            "has_vulnerable_acl": False,
            "schema_version": 2,
            "hard_negative_type": "ESC1",
        }
    elif mode == "HN_ESC4":
        # Features match ESC4, but we will block vulnerable ACL write edges
        return {
            "name_flag": random.choice([0, 0x2000000]),
            "enrollment_flag": random.choice([0, 32, 43]),
            "ra_signature": random.choice([0, 1]),
            "ekus": [random.choice([EKU_CODE_SIGNING, EKU_CLIENT_AUTH, EKU_SERVER_AUTH])],
            "has_issuance_policy_oid": False,
            "has_vulnerable_acl": True,
            "schema_version": 2,
            "hard_negative_type": "ESC4",
        }
    else:  # HN_ESC13
        # Features match ESC13, but we will block the linked issuance policy edge
        return {
            "name_flag": 0x2000000,
            "enrollment_flag": 0,
            "ra_signature": 0,
            "ekus": [EKU_CLIENT_AUTH],
            "has_issuance_policy_oid": True,
            "has_vulnerable_acl": False,
            "schema_version": 2,
            "hard_negative_type": "ESC13",
        }


TEMPLATE_GENERATORS = {
    "ESC1": _gen_esc1_template,
    "ESC2": _gen_esc2_template,
    "ESC3": _gen_esc3_template,
    "ESC4": _gen_esc4_template,
    "ESC9": _gen_esc9_template,
    "ESC13": _gen_esc13_template,
    "Safe": _gen_safe_template,
}


# ─────────────────────────────────────────────────────────────────────
# Feature extraction (matches CertGraph schema)
# ─────────────────────────────────────────────────────────────────────

def template_to_features(tmpl: dict) -> list[float]:
    """Convert template config to 10-dim feature vector."""
    nf = tmpl["name_flag"]
    ef = tmpl["enrollment_flag"]
    ekus = set(tmpl["ekus"])

    return [
        1.0 if (nf & CT_FLAG_ENROLLEE_SUPPLIES_SUBJECT) else 0.0,
        0.0 if (ef & 0x02) else 1.0,  # manager approval NOT required
        1.0 if (ef & CT_FLAG_NO_SECURITY_EXTENSION) else 0.0,
        1.0 if EKU_CLIENT_AUTH in ekus else 0.0,
        1.0 if EKU_ANY_PURPOSE in ekus else 0.0,
        1.0 if EKU_CERT_REQUEST_AGENT in ekus else 0.0,
        float(tmpl["ra_signature"]),
        1.0 if tmpl["has_issuance_policy_oid"] else 0.0,
        1.0 if tmpl["has_vulnerable_acl"] else 0.0,
        tmpl["schema_version"] / 4.0,
    ]


TEMPLATE_FEATURE_DIM = 10
TEMPLATE_FEATURE_NAMES = [
    "enrollee_supplies_subject", "no_manager_approval", "no_security_extension",
    "has_client_auth", "has_any_purpose", "has_cert_req_agent",
    "ra_signature_required", "has_issuance_policy_oid", "has_vulnerable_acl",
    "schema_version_norm",
]

USER_FEATURE_DIM = 6
GROUP_FEATURE_DIM = 2
COMPUTER_FEATURE_DIM = 3
CA_FEATURE_DIM = 3


# ─────────────────────────────────────────────────────────────────────
# Full environment generator
# ─────────────────────────────────────────────────────────────────────

def generate_environment(
    esc_class: str,
    num_users: int | None = None,
    num_groups: int | None = None,
    num_computers: int | None = None,
    num_extra_templates: int | None = None,
    seed: int | None = None,
) -> tuple[HeteroData, int]:
    """Generate a single synthetic AD environment with one target template."""
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    num_users = num_users or random.randint(10, 80)
    num_groups = num_groups or random.randint(5, 30)
    num_computers = num_computers or random.randint(1, 5)
    num_extra_templates = num_extra_templates or random.randint(0, 3)
    num_cas = 1

    data = HeteroData()

    # ── User features ──
    user_feats = []
    for i in range(num_users):
        is_admin = i < max(1, num_users // 10)
        user_feats.append([
            1.0,
            1.0 if is_admin and random.random() < 0.3 else 0.0,
            1.0 if random.random() < 0.1 else 0.0,
            1.0 if random.random() < 0.05 else 0.0,
            1.0 if random.random() < 0.15 else 0.0,
            1.0 if is_admin else 0.0,
        ])
    data["User"].x = torch.tensor(user_feats, dtype=torch.float)

    # ── Group features ──
    group_feats = []
    da_idx = 0
    for i in range(num_groups):
        is_highvalue = i < max(1, num_groups // 5)
        group_feats.append([
            1.0 if is_highvalue else 0.0,
            1.0 if is_highvalue else 0.0,
        ])
    data["Group"].x = torch.tensor(group_feats, dtype=torch.float)

    # ── Computer features ──
    comp_feats = []
    for i in range(num_computers):
        comp_feats.append([
            1.0,
            1.0 if random.random() < 0.2 else 0.0,
            1.0 if random.random() < 0.3 else 0.0,
        ])
    data["Computer"].x = torch.tensor(comp_feats, dtype=torch.float)

    # ── CA features ──
    ca_feats = [[
        1.0 if random.random() < 0.7 else 0.0,
        1.0 if random.random() < 0.3 else 0.0,
        1.0 if random.random() < 0.5 else 0.0,
    ]]
    data["CA"].x = torch.tensor(ca_feats, dtype=torch.float)

    # ── Template features ──
    target_tmpl = TEMPLATE_GENERATORS[esc_class]()
    templates = [target_tmpl]

    for _ in range(num_extra_templates):
        extra_class = random.choice(["Safe", "Safe", "Safe", esc_class])
        templates.append(TEMPLATE_GENERATORS[extra_class]())

    template_feats = [template_to_features(t) for t in templates]
    data["Template"].x = torch.tensor(template_feats, dtype=torch.float)
    num_templates = len(templates)

    # Template labels
    labels = torch.full((num_templates,), CLASS_TO_IDX["Safe"], dtype=torch.long)
    labels[0] = CLASS_TO_IDX[esc_class]
    data["Template"].y = labels

    # ── User → member_of → Group ──
    user_group_edges = []
    for u in range(max(1, num_users // 10)):
        for g in range(max(1, num_groups // 5)):
            if random.random() < 0.6:
                user_group_edges.append([u, g])
    for u in range(num_users):
        num_memberships = random.randint(1, min(4, num_groups))
        for g in random.sample(range(num_groups), num_memberships):
            user_group_edges.append([u, g])

    if user_group_edges:
        data["User", "member_of", "Group"].edge_index = (
            torch.tensor(user_group_edges, dtype=torch.long).t().contiguous()
        )
    else:
        data["User", "member_of", "Group"].edge_index = torch.empty((2, 0), dtype=torch.long)

    # ── Group → member_of → Group ──
    group_group_edges = []
    for g in range(1, num_groups):
        if random.random() < 0.2:
            parent = random.randint(0, g - 1)
            group_group_edges.append([g, parent])
    if group_group_edges:
        data["Group", "member_of", "Group"].edge_index = (
            torch.tensor(group_group_edges, dtype=torch.long).t().contiguous()
        )
    else:
        data["Group", "member_of", "Group"].edge_index = torch.empty((2, 0), dtype=torch.long)

    # ── Group → generic_all → Group ──
    ace_edges_gg = []
    for g_src in range(max(1, num_groups // 5)):
        for g_dst in range(num_groups):
            if random.random() < 0.15:
                ace_edges_gg.append([g_src, g_dst])
    if ace_edges_gg:
        data["Group", "generic_all", "Group"].edge_index = (
            torch.tensor(ace_edges_gg, dtype=torch.long).t().contiguous()
        )
    else:
        data["Group", "generic_all", "Group"].edge_index = torch.empty((2, 0), dtype=torch.long)

    # ── User → write_dacl → Template ──
    tmpl_acl_edges = []
    for t in range(num_templates):
        tmpl = templates[t]
        if tmpl["has_vulnerable_acl"]:
            is_hn_esc4 = tmpl.get("hard_negative_type") == "ESC4"
            # Real ESC4 has low-priv write DACL; HN_ESC4 does not!
            if not is_hn_esc4:
                for u in range(max(1, num_users // 10), num_users):
                    if random.random() < 0.3:
                        tmpl_acl_edges.append([u, t])
    if tmpl_acl_edges:
        data["User", "write_dacl", "Template"].edge_index = (
            torch.tensor(tmpl_acl_edges, dtype=torch.long).t().contiguous()
        )
    else:
        data["User", "write_dacl", "Template"].edge_index = torch.empty((2, 0), dtype=torch.long)

    # ── User → enrolls → Template ──
    enroll_edges = []
    for u in range(num_users):
        is_admin = u < max(1, num_users // 10)
        for t in range(num_templates):
            tmpl = templates[t]
            is_hn_esc1 = tmpl.get("hard_negative_type") == "ESC1"
            
            # Hard Negative ESC1: low-priv users CANNOT enroll (only admins can)
            if is_hn_esc1:
                if is_admin and random.random() < 0.4:
                    enroll_edges.append([u, t])
            else:
                # Normal or other ESC: standard enroll permissions
                if not is_admin and random.random() < 0.4:
                    enroll_edges.append([u, t])
                elif is_admin and random.random() < 0.2:
                    enroll_edges.append([u, t])
                    
    if enroll_edges:
        data["User", "enrolls", "Template"].edge_index = (
            torch.tensor(enroll_edges, dtype=torch.long).t().contiguous()
        )
    else:
        data["User", "enrolls", "Template"].edge_index = torch.empty((2, 0), dtype=torch.long)

    # ── Template → issued_by → CA ──
    issued_edges = [[t, 0] for t in range(num_templates)]
    data["Template", "issued_by", "CA"].edge_index = (
        torch.tensor(issued_edges, dtype=torch.long).t().contiguous()
    )

    # ── Template → linked_to → Group ──
    policy_edges = []
    for t in range(num_templates):
        tmpl = templates[t]
        if tmpl["has_issuance_policy_oid"]:
            is_hn_esc13 = tmpl.get("hard_negative_type") == "ESC13"
            # Real ESC13 has issuance policy linked to high-value group; HN_ESC13 does not!
            if not is_hn_esc13:
                policy_edges.append([t, da_idx])
    if policy_edges:
        data["Template", "linked_to", "Group"].edge_index = (
            torch.tensor(policy_edges, dtype=torch.long).t().contiguous()
        )
    else:
        data["Template", "linked_to", "Group"].edge_index = torch.empty((2, 0), dtype=torch.long)

    # ── Computer → member_of → Group ──
    comp_group_edges = [[c, random.randint(0, num_groups - 1)] for c in range(num_computers)]
    data["Computer", "member_of", "Group"].edge_index = (
        torch.tensor(comp_group_edges, dtype=torch.long).t().contiguous()
    )

    return data, 0


def generate_dataset(
    num_envs: int = 500,
    balanced: bool = True,
    seed: int = 42,
) -> list[tuple[HeteroData, str, int]]:
    """Generate a full labeled dataset of synthetic AD environments."""
    random.seed(seed)
    np.random.seed(seed)

    dataset = []
    if balanced:
        per_class = num_envs // NUM_CLASSES
        remainder = num_envs % NUM_CLASSES
        class_counts = {c: per_class for c in ESC_CLASSES}
        for i, c in enumerate(ESC_CLASSES):
            if i < remainder:
                class_counts[c] += 1
    else:
        class_counts = {c: num_envs // NUM_CLASSES for c in ESC_CLASSES}

    env_id = 0
    for esc_class, count in class_counts.items():
        for _ in range(count):
            data, target_idx = generate_environment(
                esc_class=esc_class,
                seed=seed + env_id,
            )
            dataset.append((data, esc_class, target_idx))
            env_id += 1

    random.shuffle(dataset)
    return dataset


## 4. CertGraph Model + Baselines


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, HeteroConv
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier


# ─────────────────────────────────────────────────────────────────────
# CertGraph — Main Model
# ─────────────────────────────────────────────────────────────────────

class CertGraph(nn.Module):
    """
    Certificate-Aware Heterogeneous Graph Attention Network with Residual Skip Connections.

    Performs multi-class node classification on Template nodes to predict
    their ESC vulnerability class (ESC1/ESC2/ESC3/ESC4/ESC9/ESC13/Safe).
    """

    def __init__(
        self,
        metadata: tuple,
        hidden_channels: int = 32,
        out_channels: int = 16,
        num_classes: int = 7,
        num_heads: int = 4,
        dropout: float = 0.2,
        skip_connections: bool = True,
    ):
        super().__init__()
        self.dropout = dropout
        self.num_classes = num_classes
        self.node_types = metadata[0]
        self._hidden_total = hidden_channels * num_heads
        self._out_channels = out_channels
        self.skip_connections = skip_connections

        # Layer 1: Multi-head GAT per relation type
        self.conv1 = HeteroConv(
            {
                edge_type: GATConv(
                    (-1, -1), hidden_channels, heads=num_heads,
                    add_self_loops=False, dropout=dropout,
                )
                for edge_type in metadata[1]
            },
            aggr="sum",
        )

        # Layer 2: Single-head GAT
        self.conv2 = HeteroConv(
            {
                edge_type: GATConv(
                    hidden_channels * num_heads, out_channels, heads=1,
                    concat=False, add_self_loops=False, dropout=dropout,
                )
                for edge_type in metadata[1]
            },
            aggr="sum",
        )

        # Lazy skip connection layers — will be initialized on first forward pass
        self._projections_initialized = False
        self.skip1 = nn.ModuleDict()
        self.skip2 = nn.ModuleDict()

        # Classification head for Template nodes
        self.classifier = nn.Sequential(
            nn.Linear(out_channels, out_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(out_channels, num_classes),
        )

    def _ensure_projections(self, x_dict: dict[str, torch.Tensor]):
        """Create skip connection projection layers on first pass."""
        if self._projections_initialized:
            return
        for ntype, x in x_dict.items():
            self.skip1[ntype] = nn.Linear(x.shape[1], self._hidden_total)
            self.skip2[ntype] = nn.Linear(self._hidden_total, self._out_channels)
        self._projections_initialized = True

    def forward(
        self,
        x_dict: dict[str, torch.Tensor],
        edge_index_dict: dict[tuple, torch.Tensor],
    ) -> torch.Tensor:
        """
        Forward pass. Returns logits for Template nodes.

        Returns:
            Tensor of shape [num_templates, num_classes]
        """
        self._ensure_projections(x_dict)

        # Message passing layer 1
        x_dict_1 = self.conv1(x_dict, edge_index_dict)
        
        # Apply Layer 1 skip connections (ResNet residual block)
        out_dict_1 = {}
        for k, v in x_dict.items():
            proj_val = self.skip1[k](v)
            if k in x_dict_1 and x_dict_1[k] is not None:
                if self.skip_connections:
                    out_dict_1[k] = x_dict_1[k] + proj_val
                else:
                    out_dict_1[k] = x_dict_1[k]
            else:
                out_dict_1[k] = proj_val
                
        out_dict_1 = {k: F.relu(v) for k, v in out_dict_1.items()}
        out_dict_1 = {
            k: F.dropout(v, p=self.dropout, training=self.training)
            for k, v in out_dict_1.items()
        }

        # Message passing layer 2
        x_dict_2 = self.conv2(out_dict_1, edge_index_dict)
        
        # Apply Layer 2 skip connections
        out_dict_2 = {}
        for k, v in out_dict_1.items():
            proj_val = self.skip2[k](v)
            if k in x_dict_2 and x_dict_2[k] is not None:
                if self.skip_connections:
                    out_dict_2[k] = x_dict_2[k] + proj_val
                else:
                    out_dict_2[k] = x_dict_2[k]
            else:
                out_dict_2[k] = proj_val

        # Classify Template nodes
        template_emb = out_dict_2["Template"]
        logits = self.classifier(template_emb)
        return logits

    def get_embeddings(
        self,
        x_dict: dict[str, torch.Tensor],
        edge_index_dict: dict[tuple, torch.Tensor],
    ) -> dict[str, torch.Tensor]:
        """Get node embeddings from layer 2 (before classifier)."""
        self._ensure_projections(x_dict)
        
        x_dict_1 = self.conv1(x_dict, edge_index_dict)
        out_dict_1 = {}
        for k, v in x_dict.items():
            proj_val = self.skip1[k](v)
            if k in x_dict_1 and x_dict_1[k] is not None:
                if self.skip_connections:
                    out_dict_1[k] = x_dict_1[k] + proj_val
                else:
                    out_dict_1[k] = x_dict_1[k]
            else:
                out_dict_1[k] = proj_val
        out_dict_1 = {k: F.relu(v) for k, v in out_dict_1.items()}
        
        x_dict_2 = self.conv2(out_dict_1, edge_index_dict)
        out_dict_2 = {}
        for k, v in out_dict_1.items():
            proj_val = self.skip2[k](v)
            if k in x_dict_2 and x_dict_2[k] is not None:
                if self.skip_connections:
                    out_dict_2[k] = x_dict_2[k] + proj_val
                else:
                    out_dict_2[k] = x_dict_2[k]
            else:
                out_dict_2[k] = proj_val
                
        return out_dict_2

    def get_attention_weights(
        self,
        x_dict: dict[str, torch.Tensor],
        edge_index_dict: dict[tuple, torch.Tensor],
    ) -> dict[tuple, tuple[torch.Tensor, torch.Tensor]]:
        """Extract attention weights for conv1."""
        self._ensure_projections(x_dict)
        attentions = {}
        for edge_type, edge_index in edge_index_dict.items():
            src, rel, dst = edge_type
            h_src = x_dict[src]
            h_dst = x_dict[dst]
            conv = self.conv1.convs[edge_type]
            _, (edge_index_out, alpha) = conv((h_src, h_dst), edge_index, return_attention_weights=True)
            attentions[edge_type] = (edge_index_out, alpha)
        return attentions


# ─────────────────────────────────────────────────────────────────────
# Baseline 1: MLP on flat template features
# ─────────────────────────────────────────────────────────────────────

class MLPBaseline(nn.Module):
    """MLP classifier using only template features (no graph structure)."""

    def __init__(self, input_dim: int = 10, hidden_dim: int = 32, num_classes: int = 7):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ─────────────────────────────────────────────────────────────────────
# Baseline 2: Rule-based (Certipy-style heuristic)
# ─────────────────────────────────────────────────────────────────────

class RuleBaseline:
    """
    Deterministic rule-based classifier mimicking Certipy/Certify logic.
    Uses the 10-dim template feature vector directly.

    Feature indices:
        0: enrollee_supplies_subject
        1: no_manager_approval
        2: no_security_extension
        3: has_client_auth
        4: has_any_purpose
        5: has_cert_req_agent
        6: ra_signature_required
        7: has_issuance_policy_oid
        8: has_vulnerable_acl
        9: schema_version_norm
    """

    def predict(self, features: torch.Tensor) -> torch.Tensor:
        """
        Classify templates based on deterministic rules.

        Args:
            features: [num_templates, 10] feature tensor

        Returns:
            [num_templates] predicted class indices
        """
        preds = []
        for i in range(features.shape[0]):
            f = features[i]
            ess = f[0].item()     # enrollee_supplies_subject
            nma = f[1].item()     # no_manager_approval
            nse = f[2].item()     # no_security_extension
            ca = f[3].item()      # has_client_auth
            ap = f[4].item()      # has_any_purpose
            cra = f[5].item()     # has_cert_req_agent
            ras = f[6].item()     # ra_signature_required
            ipo = f[7].item()     # has_issuance_policy_oid
            vacl = f[8].item()    # has_vulnerable_acl

            # Priority ordering matters
            if ess > 0 and ca > 0 and nma > 0 and ras < 0.5:
                preds.append(0)  # ESC1
            elif ap > 0 and nma > 0:
                preds.append(1)  # ESC2
            elif (cra > 0 or ras > 0) and nma > 0:
                preds.append(2)  # ESC3
            elif vacl > 0:
                preds.append(3)  # ESC4
            elif nse > 0 and ca > 0:
                preds.append(4)  # ESC9
            elif ipo > 0 and ca > 0:
                preds.append(5)  # ESC13
            else:
                preds.append(6)  # Safe

        return torch.tensor(preds, dtype=torch.long)


## 5. Generate & Inspect Dataset


In [ ]:
NUM_ENVS = 700
dataset = generate_dataset(num_envs=NUM_ENVS, balanced=True, seed=42)

# Add custom attributes for PyG DataLoader batching
for data, esc_class, target_idx in dataset:
    data.target_idx = torch.tensor([target_idx], dtype=torch.long)
    data.y_class = torch.tensor([CLASS_TO_IDX[esc_class]], dtype=torch.long)

class_counts = Counter(c for _, c, _ in dataset)
print(f"Generated {len(dataset)} environments with adversarial hard negatives:")
for cls in ESC_CLASSES:
    print(f"  {cls}: {class_counts[cls]}")

# Inspect one sample
sample, cls, idx = dataset[0]
print(f"\nSample (class={cls}):")
for nt in sample.node_types:
    print(f"  {nt}: {sample[nt].x.shape}")
for et in sample.edge_types:
    print(f"  {et[0]}--{et[1]}-->{et[2]}: {sample[et].edge_index.shape[1]}")


## 6. 5-Fold Cross-Validation Training


In [ ]:
# Import baselines from train.py module
from train import train_mlp_fold, eval_rule_baseline, train_rf_fold

# ── Hyperparameters ──
HIDDEN_DIM, OUT_DIM, NUM_HEADS, DROPOUT = 32, 16, 4, 0.2
LR, EPOCHS, N_FOLDS, BATCH_SIZE = 0.005, 100, 5, 64

labels_arr = [CLASS_TO_IDX[c] for _, c, _ in dataset]
indices = np.arange(len(dataset))
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

results = {n: defaultdict(list) for n in ["CertGraph", "MLP", "Rule-Based", "Random Forest"]}
all_histories = []
best_model, best_f1 = None, 0
last_cg_results = None

for fold, (train_idx, test_idx) in enumerate(skf.split(indices, labels_arr)):
    print(f"\n── Fold {fold+1}/{N_FOLDS} ──")
    train_data = [dataset[i] for i in train_idx]
    test_data = [dataset[i] for i in test_idx]

    # ── CertGraph ──
    print("  [CertGraph]")
    s0 = train_data[0][0]
    model = CertGraph(metadata=s0.metadata(), hidden_channels=HIDDEN_DIM,
                      out_channels=OUT_DIM, num_classes=NUM_CLASSES,
                      num_heads=NUM_HEADS, dropout=DROPOUT)
    with torch.no_grad():
        model.eval(); _ = model(s0.x_dict, s0.edge_index_dict)
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    history = {"loss": [], "train_f1": []}

    train_list = [d for d, _, _ in train_data]
    test_list = [d for d, _, _ in test_data]
    train_loader = DataLoader(train_list, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_list, batch_size=BATCH_SIZE, shuffle=False)

    for epoch in range(1, EPOCHS+1):
        model.train(); total_loss = 0; preds_e, labs_e = [], []
        for batch in train_loader:
            optimizer.zero_grad()
            logits = model(batch.x_dict, batch.edge_index_dict)
            ptr = batch["Template"].ptr
            target_indices = ptr[:-1] + batch.target_idx
            target_logits = logits[target_indices]
            loss = F.cross_entropy(target_logits, batch.y_class)
            loss.backward(); optimizer.step()
            total_loss += loss.item() * batch.num_graphs
            preds_e.extend(target_logits.argmax(dim=1).cpu().tolist())
            labs_e.extend(batch.y_class.cpu().tolist())
        history["loss"].append(total_loss/len(train_list))
        history["train_f1"].append(f1_score(labs_e, preds_e, average="macro", zero_division=0))
        if epoch % 25 == 0:
            print(f"    Epoch {epoch:03d} | Loss: {history['loss'][-1]:.4f} | F1: {history['train_f1'][-1]:.4f}")

    model.eval(); tp, tl = [], []
    with torch.no_grad():
        for batch in test_loader:
            logits = model(batch.x_dict, batch.edge_index_dict)
            ptr = batch["Template"].ptr
            target_indices = ptr[:-1] + batch.target_idx
            target_logits = logits[target_indices]
            tp.extend(target_logits.argmax(dim=1).cpu().tolist())
            tl.extend(batch.y_class.cpu().tolist())
    cg_f1 = f1_score(tl, tp, average="macro", zero_division=0)
    cg_acc = accuracy_score(tl, tp)
    results["CertGraph"]["f1"].append(cg_f1)
    results["CertGraph"]["acc"].append(cg_acc)
    all_histories.append(history)
    if cg_f1 > best_f1: best_f1, best_model = cg_f1, model
    last_cg_results = {"preds": tp, "labels": tl}
    print(f"    CertGraph → F1: {cg_f1:.4f} | Acc: {cg_acc:.4f}")

    # ── MLP baseline ──
    mlp_res = train_mlp_fold(train_data, test_data)
    results["MLP"]["f1"].append(mlp_res["test_f1"])
    results["MLP"]["acc"].append(mlp_res["test_acc"])
    print(f"  MLP       → F1: {mlp_res['test_f1']:.4f}")

    # ── Rule-based baseline ──
    rule_res = eval_rule_baseline(test_data)
    results["Rule-Based"]["f1"].append(rule_res["test_f1"])
    results["Rule-Based"]["acc"].append(rule_res["test_acc"])
    print(f"  Rule      → F1: {rule_res['test_f1']:.4f}")

    # ── Random Forest baseline ──
    rf_res = train_rf_fold(train_data, test_data)
    results["Random Forest"]["f1"].append(rf_res["test_f1"])
    results["Random Forest"]["acc"].append(rf_res["test_acc"])
    print(f"  RF        → F1: {rf_res['test_f1']:.4f}")

print("\n" + "="*60)
print(f"{'Model':<20} {'Macro-F1':>12} {'Accuracy':>12}")
print("-" * 46)
for name, m in results.items():
    print(f"{name:<20} {np.mean(m['f1']):.4f}±{np.std(m['f1']):.4f} {np.mean(m['acc']):.4f}±{np.std(m['acc']):.4f}")


## 7. Results & Visualization

### 7.1 Training Curves


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for i, h in enumerate(all_histories):
    ax1.plot(h["loss"], alpha=0.4, label=f"Fold {i+1}")
    ax2.plot(h["train_f1"], alpha=0.4, label=f"Fold {i+1}")
ax1.set_title("Training Loss", fontsize=13, fontweight="bold")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("CE Loss"); ax1.legend(facecolor="#0e1628", edgecolor="#333", labelcolor="white")
ax1.grid(True, alpha=0.1, color="#555")
ax2.set_title("Training Macro-F1", fontsize=13, fontweight="bold")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("F1"); ax2.legend(facecolor="#0e1628", edgecolor="#333", labelcolor="white")
ax2.set_ylim(0, 1.05); ax2.grid(True, alpha=0.1, color="#555")
fig.suptitle("CertGraph Training — 5-Fold CV", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()


### 7.2 Baseline Comparison


In [ ]:
model_names = list(results.keys())
f1_means = [np.mean(results[n]["f1"]) for n in model_names]
f1_stds = [np.std(results[n]["f1"]) for n in model_names]
colors = ["#FF6B6B", "#4ECDC4", "#FFA07A", "#45B7D1"]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(model_names, f1_means, yerr=f1_stds, color=colors, alpha=0.85,
              edgecolor="white", linewidth=0.5, capsize=5)
for bar, val in zip(bars, f1_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{val:.3f}", ha="center", va="bottom", fontsize=12, fontweight="bold", color="white")
ax.set_ylabel("Macro-F1 Score", fontsize=12)
ax.set_title("Model Comparison — ESC Vulnerability Classification", fontsize=14, fontweight="bold", pad=15)
ax.set_ylim(0, 1.15)
for s in ax.spines.values(): s.set_color("#333")
ax.grid(True, axis="y", alpha=0.1, color="#555")
plt.tight_layout(); plt.show()


### 7.3 Confusion Matrix (Best CertGraph Fold)


In [ ]:
if last_cg_results:
    cm = sk_confusion_matrix(last_cg_results["labels"], last_cg_results["preds"],
                             labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm, cmap="YlOrRd", aspect="auto")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(ESC_CLASSES, rotation=45, ha="right", color="white")
    ax.set_yticklabels(ESC_CLASSES, color="white")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            c = "white" if cm[i,j] > cm.max()/2 else "black"
            ax.text(j, i, str(cm[i,j]), ha="center", va="center", color=c, fontweight="bold")
    ax.set_xlabel("Predicted", fontsize=12); ax.set_ylabel("True", fontsize=12)
    ax.set_title("CertGraph Confusion Matrix", fontsize=14, fontweight="bold", pad=15)
    plt.colorbar(im, ax=ax)
    plt.tight_layout(); plt.show()

    print(classification_report(last_cg_results["labels"], last_cg_results["preds"],
          target_names=ESC_CLASSES, zero_division=0))


### 7.4 t-SNE Embedding Visualization


In [ ]:
if best_model:
    # Collect embeddings from test samples
    all_emb, all_lab = [], []
    best_model.eval()
    with torch.no_grad():
        for data, ec, ti in dataset[:150]:
            emb_dict = best_model.get_embeddings(data.x_dict, data.edge_index_dict)
            if "Template" in emb_dict and emb_dict["Template"] is not None:
                all_emb.append(emb_dict["Template"][ti].cpu().numpy())
                all_lab.append(ec)

    if all_emb:
        emb_arr = np.array(all_emb)
        perp = min(30, max(2, len(emb_arr)-1))
        coords = TSNE(n_components=2, random_state=42, perplexity=perp, max_iter=1000).fit_transform(emb_arr)
        clrs = {"ESC1":"#FF6B6B","ESC2":"#4ECDC4","ESC3":"#FFA07A","ESC4":"#45B7D1",
                "ESC9":"#E8A0BF","ESC13":"#FFD93D","Safe":"#95E1D3"}
        fig, ax = plt.subplots(figsize=(10, 8))
        for i, (x, y) in enumerate(coords):
            ax.scatter(x, y, c=clrs.get(all_lab[i], "#fff"), s=60, alpha=0.8, edgecolors="white", linewidths=0.3)
        legend_el = [Line2D([0],[0], marker="o", color="w", markerfacecolor=clrs[c],
                     markersize=10, label=c, ls="None") for c in ESC_CLASSES if c in clrs]
        ax.legend(handles=legend_el, loc="upper right", fontsize=10, facecolor="#0e1628", edgecolor="#333", labelcolor="white")
        ax.set_title("CertGraph Template Embeddings — t-SNE", fontsize=14, fontweight="bold", pad=15)
        for s in ax.spines.values(): s.set_color("#333")
        plt.tight_layout(); plt.show()


## 8. Advanced Academic Evaluations (Q1/Peer-Review Enhancements)

To establish scientific rigor, interpretability, and practical scalability, we evaluate CertGraph under four advanced experimental protocols:

1. **Ablation Studies**: Evaluating key architectural configurations (Residual Skip Connections, Graph structure, Attention mechanisms).
2. **GNN Explainability**: Extracting and plotting GAT attention coefficients to prove the model attends to valid security-relevant relationships.
3. **Scalability Testing**: Benchmarking training/inference latency and memory consumption on AD graphs scaled up to 10,000 nodes.
4. **Adversarial Hard Negative Evaluation**: Assessing performance on safe templates designed to mimic vulnerabilities via configuration flags but lacking exploitable graph relationships.


### 8.1 Ablation Study Results

We compare the macro-F1 and accuracy of the full model against:
- **No Skip**: Disables the ResNet-style residual projection skip connections.
- **Single-Head**: Uses single-head GAT attention (parameter matched).
- **No Graph**: Zeroes out all relationship edges, reducing representation learning to node properties.

*Note: These results are pre-computed by running the ablation pipeline (`src/phase2_certgraph/ablation.py`).*


In [ ]:
import os, json
from IPython.display import Image, display
results_dir = "../results/phase2"
ablation_json = os.path.join(results_dir, "ablation_results.json")
ablation_plot = os.path.join(results_dir, "ablation_comparison.png")

if os.path.exists(ablation_json):
    with open(ablation_json) as f:
        data = json.load(f)
    print(f"{chr(61)*62}")
    print(f"{'Variant':<26} {'Macro-F1':>12} {'Accuracy':>12}")
    print(f"{chr(45)*62}")
    for var, metrics in data.items():
        print(f"{var:<26} {metrics['f1_mean']:.4f}±{metrics['f1_std']:.4f} {metrics['acc_mean']:.4f}±{metrics['acc_std']:.4f}")
    print(f"{chr(61)*62}")
else:
    print("[!] Ablation results JSON not found. Please run ablation.py to populate.")

if os.path.exists(ablation_plot):
    display(Image(filename=ablation_plot))


### 8.2 Explainability & Attention Analysis

GNN interpretability is evaluated by extracting the attention coefficients ($\alpha_{ij}$) from GATConv layer 1. We plot the top edges that received the highest attention weights when predicting an ESC13 vulnerability, demonstrating that the GNN utilizes domain-specific AD security relationships rather than local node heuristics.

*Note: These results are pre-computed by running the explainability pipeline (`src/phase2_certgraph/explain.py`).*


In [ ]:
explain_plot = os.path.join(results_dir, "attention_explainability.png")
if os.path.exists(explain_plot):
    display(Image(filename=explain_plot))
else:
    print("[!] Explainability plot not found. Please run explain.py to populate.")


### 8.3 Scalability & Performance Benchmarks

We evaluate how CertGraph scales to massive enterprise networks by measuring inference latency (ms), peak memory (MB), and generation time across AD graphs ranging from 100 nodes to 10,000 nodes.

*Note: These results are pre-computed by running the scalability pipeline (`src/phase2_certgraph/scale_test.py`).*


In [ ]:
scale_json = os.path.join(results_dir, "scalability_results.json")
scale_plot = os.path.join(results_dir, "scalability_metrics.png")

if os.path.exists(scale_json):
    with open(scale_json) as f:
        scales = json.load(f)
    print(f"{chr(61)*75}")
    print(f"{'Scale (Nodes)':<15} {'Edges':>12} {'Gen Time (ms)':>15} {'Memory (MB)':>12} {'Inference (ms)':>15}")
    print(f"{chr(45)*75}")
    for r in scales:
        print(f"{r['actual_nodes']:<15,} {r['actual_edges']:>12,} {r['generation_time_ms']:>15.1f} {r['memory_usage_mb']:>12.3f} {r['inference_latency_ms_mean']:>15.2f}")
    print(f"{chr(61)*75}")
else:
    print("[!] Scalability results JSON not found. Please run scale_test.py to populate.")

if os.path.exists(scale_plot):
    display(Image(filename=scale_plot))


### 8.4 Adversarial Hard Negative Performance

Finally, we prove that flat machine learning classifiers (MLP, RF) and heuristic rule-based systems are easily fooled by AD objects configured to look like vulnerabilities (e.g. templates with the enrollee supplies subject flag enabled) but lacking the corresponding enrollment permissions or DACL write relationships. CertGraph correctly routes updates and identifies these templates as Safe using relation-specific message passing.

*Note: These results are pre-computed by running the hard negative pipeline (`src/phase2_certgraph/hard_negatives.py`).*


In [ ]:
hn_json = os.path.join(results_dir, "hard_negatives_results.json")
hn_plot = os.path.join(results_dir, "hard_negatives_comparison.png")

if os.path.exists(hn_json):
    with open(hn_json) as f:
        data = json.load(f)
    print(f"{chr(61)*52}")
    print(f"{'Model':<25} {'Accuracy on Hard Negatives':>25}")
    print(f"{chr(45)*52}")
    for model_name, acc in data.items():
        print(f"{model_name:<25} {acc*100:>23.2f}%")
    print(f"{chr(61)*52}")
else:
    print("[!] Hard negatives results JSON not found. Please run hard_negatives.py to populate.")

if os.path.exists(hn_plot):
    display(Image(filename=hn_plot))


### 8.5 Domain Generalization & Transfer Learning (ADSynth)

To demonstrate domain generalization, we train CertGraph on a standard synthetic topology generator and test on realistic, tiered topologies (following Microsoft's admin tiering model generated by ADSynth), and vice versa. We also establish a GNN baseline by running a 5-fold cross-validation solely on the tiered ADSynth environments.

*Note: These results are pre-computed by running the transfer learning pipeline (`src/phase2_certgraph/transfer_eval.py`).*


In [ ]:
transfer_json = os.path.join(results_dir, "transfer_results.json")
if os.path.exists(transfer_json):
    with open(transfer_json) as f:
        tr = json.load(f)
    print(f"{chr(61)*75}")
    print("DOMAIN TRANSFER EXPERIMENTS")
    print(f"{chr(45)*75}")
    print(f"  - Train Synthetic -> Test ADSynth:  Macro-F1 = {tr['synthetic_to_adsynth']['f1']:.4f} | Acc = {tr['synthetic_to_adsynth']['accuracy']:.4f}")
    print(f"  - Train ADSynth -> Test Synthetic:  Macro-F1 = {tr['adsynth_to_synthetic']['f1']:.4f} | Acc = {tr['adsynth_to_synthetic']['accuracy']:.4f}")
    print(f"{chr(45)*75}")
    print("ADSYNTH (REALISTIC TOPOLOGY) 5-FOLD CROSS-VALIDATION")
    print(f"  - Mean Macro F1-Score: {tr['adsynth_5fold_cv']['f1_mean']:.4f}±{tr['adsynth_5fold_cv']['f1_std']:.4f}")
    print(f"  - Mean Accuracy:        {tr['adsynth_5fold_cv']['acc_mean']:.4f}±{tr['adsynth_5fold_cv']['acc_std']:.4f}")
    print(f"{chr(61)*75}")
else:
    print("[!] Transfer results JSON not found. Please run transfer_eval.py to populate.")


### 8.6 Multi-Tool Comparison Baseline

We compare CertGraph against two security assessment paradigms:
1. **Certipy (Heuristic Rules)**: Assesses templates purely by local configuration flags, leading to high false positives on structurally blocked attack paths.
2. **BloodHound (Explicit Queries)**: Uses manual graph traversals which fail to represent GNN soft probabilities or complex composite relationships.

*Note: These results are pre-computed by running the comparison pipeline (`src/phase2_certgraph/tool_comparison.py` and `plot_comparison.py`).*


In [ ]:
comp_json = os.path.join(results_dir, "tool_comparison_results.json")
comp_plot = os.path.join(results_dir, "tool_comparison_f1.png")
if os.path.exists(comp_json):
    with open(comp_json) as f:
        comp = json.load(f)
    for ds, tools_res in comp.items():
        print(f"\nDataset: {ds}")
        print(f"{chr(45)*55}")
        print(f"{'Tool':<15} {'Accuracy':>10} {'Precision':>12} {'Recall':>10} {'Macro-F1':>10}")
        print(f"{chr(45)*55}")
        for t, metrics in tools_res.items():
            print(f"{t:<15} {metrics['accuracy']:.4f} {metrics['precision']:.4f} {metrics['recall']:.4f} {metrics['macro_f1']:.4f}")
        print(f"{chr(45)*55}")
else:
    print("[!] Comparison results JSON not found.")

if os.path.exists(comp_plot):
    display(Image(filename=comp_plot))


### 8.7 GNN Sensitivity & Robustness Analysis

To prove GNN resilience against adversarial attacks and network collection noise, we evaluate model predictions under:
- **Edge Perturbation**: Randomly dropping up to 30% of relationships.
- **Feature Noise**: Randomly flipping up to 20% of configuration flags.

*Note: These results are pre-computed by running the robustness pipeline (`src/phase2_certgraph/robustness.py`).*


In [ ]:
robust_json = os.path.join(results_dir, "robustness_results.json")
robust_plot = os.path.join(results_dir, "robustness_analysis.png")
if os.path.exists(robust_json):
    with open(robust_json) as f:
        rb = json.load(f)
    print("SENSITIVITY ANALYSIS METRICS:")
    print(f"  - Edge Drop F1  (0% -> 30%): {[f'{val:.4f}' for val in rb['edge_perturbation']['f1s']]}")
    print(f"  - Feature Flip F1 (0% -> 20%): {[f'{val:.4f}' for val in rb['feature_noise']['f1s']]}")
else:
    print("[!] Robustness results JSON not found.")

if os.path.exists(robust_plot):
    display(Image(filename=robust_plot))


## 9. Real-World AD Topology Case Studies

### 9.1 Local GOAD Active Directory Topology

We validate CertGraph by running the trained model on real-world Active Directory data parsed from SharpHound v5 files of our local **Game of Active Directory (GOAD)** training lab. We inject the vulnerability templates directly into the real user/group/computer topology and compare CertGraph predictions against Certipy heuristic rules.

*Note: These results are pre-computed by running the case study pipeline (`src/phase2_certgraph/case_study.py`).*


In [ ]:
case_json = os.path.join(results_dir, "case_study_report.json")
if os.path.exists(case_json):
    with open(case_json) as f:
        case_data = json.load(f)
    print(f"{chr(61)*82}")
    print("LOCAL GOAD TOPOLOGY EVALUATION REPORT")
    print(f"{chr(45)*82}")
    print(f"{'Test Case':<15} {'Ground Truth':<15} {'CertGraph Pred':<20} {'Rule Pred':<20}")
    print(f"{chr(45)*82}")
    for r in case_data:
        print(f"{r['case_name']:<15} {r['ground_truth']:<15} {r['certgraph_pred']:<20} {r['rule_pred']:<20}")
    print(f"{chr(61)*82}")
else:
    print("[!] Case study report JSON not found.")


### 9.2 External Community-Provided Dataset (m4lwhere)

To prove external generalization and remove collection bias, we evaluate CertGraph on a completely external dataset compiled and published by the community (from `m4lwhere/Bloodhound-CE-Sample-Data` on GitHub). This dataset represents a separate instance of the GOADv2 environment collected on different dates, platforms, and domains (`sevenkingdoms`, `essos`, and `north`).

*Note: These results are pre-computed by running the community dataset evaluation (`src/phase2_certgraph/community_eval.py`).*


In [ ]:
comm_json = os.path.join(results_dir, "community_case_study_report.json")
if os.path.exists(comm_json):
    with open(comm_json) as f:
        comm_data = json.load(f)
    for domain, cases in comm_data.items():
        print(f"\nDomain: {domain}")
        print(f"{chr(61)*80}")
        print(f"{'Test Case':<15} {'Ground Truth':<15} {'CertGraph Pred':<20} {'Rule Pred':<20}")
        print(f"{chr(45)*80}")
        correct_cg = 0
        correct_rule = 0
        for r in cases:
            print(f"{r['case_name']:<15} {r['ground_truth']:<15} {r['certgraph_pred']:<20} {r['rule_pred']:<20}")
            if r['cg_correct']: correct_cg += 1
            if r['rule_correct']: correct_rule += 1
        print(f"{chr(45)*80}")
        print(f"Accuracy: CertGraph = {correct_cg}/{len(cases)} ({correct_cg/len(cases)*100:.1f}%) | Heuristics = {correct_rule}/{len(cases)} ({correct_rule/len(cases)*100:.1f}%)")
        print(f"{chr(61)*80}")
else:
    print("[!] Community case study report JSON not found. Run community_eval.py first.")


## 10. Defense Recommendations

Based on CertGraph analysis:

| ESC | Mitigation | Effect |
|-----|-----------|--------|
| ESC1 | Disable "Enrollee Supplies Subject" | Removes impersonation capability |
| ESC2 | Remove "Any Purpose" EKU | Restricts certificate usage |
| ESC3 | Restrict Certificate Request Agent enrollment | Prevents enrollment-on-behalf |
| ESC4 | Remove low-priv WriteDacl/GenericAll on templates | Prevents template modification |
| ESC9 | Enable security extension on templates | Enforces proper cert mapping |
| ESC13 | Remove OID group link from issuance policy | Eliminates group impersonation |
| ALL  | Enable Manager Approval on sensitive templates | Human gate for all issuance |


## 11. Conclusion

CertGraph demonstrates that a **certificate-aware heterogeneous GNN** can:
- Classify 7 distinct ESC vulnerability types with high accuracy
- Outperform flat-feature baselines (MLP, RF) by leveraging graph topology
- Provide interpretable attention over AD relationships
- Generalize across synthetic AD environments of varying complexity

**Key Finding**: Graph structure matters — the enrollment, ACL, and issuance policy edges
provide critical context that template features alone cannot capture.
